# Lab 4-01: Parameter-Efficient Fine-Tuning (LoRA) with Experiment Tracking

**Chapter 4: Experimentation, Fine-Tuning, and Model Evaluation**  
**Lab Type:** Notebook Lab (GPU with Full CPU Support) | **Estimated Time:** 60 minutes  
**Exam Domains Covered:** Experimentation (22%), Core ML & AI Knowledge (30%)

---

## Learning Objectives
By the end of this lab, you will be able to:
1. Contrast **Full Fine-Tuning** vs. **Parameter-Efficient Fine-Tuning (PEFT)** and explain the low-rank matrix decomposition behind **LoRA (Low-Rank Adaptation)** and **QLoRA**.
2. Configure Hugging Face `peft` (`LoraConfig`, `r`, `lora_alpha`, `lora_dropout`, `target_modules`) and calculate exact **trainable parameter counts** ($r \times (d_{\text{in}} + d_{\text{out}})$).
3. Fine-tune a small model with a fixed random seed (`set_seed(42)`) across multiple **LoRA ranks ($r \in \{4, 8, 16\}$)** and **learning rates**, logging every run's hyperparameters, metrics, and artifacts in an **MLflow-compatible experiment tracker**.
4. Compare **Base Model vs. LoRA-Tuned Model** validation loss, perplexity, and generated responses on held-out prompts.

---
## 1. Conceptual Foundation: Why LoRA & QLoRA Changed LLM Fine-Tuning

In **Chapter 4 (*Fine-Tuning Strategies: Full Fine-Tuning vs. PEFT*)**, the book explains why updating all $N$ parameters of a 7B–70B LLM (**Full Fine-Tuning**) is often impractical:
1. **Massive VRAM Requirement:** As we calculated in Lab 2-02, full mixed-precision AdamW training requires $\sim 16\text{ bytes per parameter}$ ($\sim 112\text{ GB+}$ for a 7B model) because every trainable parameter needs gradients and FP32 Adam moments ($m_t, v_t$).
2. **Catastrophic Forgetting:** Updating all weights on a small domain dataset can overwrite general reasoning capabilities learned during pretraining.
3. **Storage & Serving Overhead:** Saving 10 task-specific 7B models requires $10 \times 14\text{ GB} = 140\text{ GB}$ of disk/VRAM.

### How Low-Rank Adaptation (LoRA) Works
Instead of modifying the pretrained weight matrix $W_0 \in \mathbb{R}^{d_{\text{out}} \times d_{\text{in}}}$, **LoRA freezes $W_0$ completely** (`requires_grad=False`) and injects a trainable low-rank update $\Delta W = B A$:
$$h = W_0 x + \frac{\alpha}{r} B A x, \quad A \in \mathbb{R}^{r \times d_{\text{in}}}, \quad B \in \mathbb{R}^{d_{\text{out}} \times r}, \quad r \ll \min(d_{\text{in}}, d_{\text{out}})$$

| LoRA Component / Hyperparameter | Meaning & Exam Significance |
| :--- | :--- |
| **`r` (Rank)** | Inner dimension of matrices $A$ and $B$ (typically `4`, `8`, `16`, or `32`). Controls the number of trainable parameters: $N_{\text{LoRA}} = r \times (d_{\text{in}} + d_{\text{out}})$ |
| **`lora_alpha` ($\alpha$)** | Scaling factor applied to the adapter output as $\frac{\alpha}{r} B A x$ (commonly set to $1\times$ or $2\times$ the rank $r$) |
| **Initialization of $A$ and $B$** | **$A \sim \mathcal{N}(0, \sigma^2)$** (Gaussian noise) and **$B = 0$** so that at step 0, $\Delta W = B A = 0$ and the model behaves identically to the pretrained base model! |
| **`target_modules`** | Which Transformer linear projections receive LoRA adapters (typically attention `q_proj`, `v_proj`, and optionally `k_proj`, `o_proj`, and MLP layers) |
| **Zero Inference Latency Penalty** | After training, LoRA weights can be **merged** directly into the base weights ($W_{\text{merged}} = W_0 + \frac{\alpha}{r}BA$), adding **0 ms of extra inference latency**! |
| **QLoRA** | Quantizes the frozen base weights $W_0$ to **4-bit NormalFloat (`NF4`)** with double quantization while computing forward/backward passes through the LoRA adapters in **BF16/FP16** |

In [1]:
import os
# Prevent Windows Anaconda + PyTorch OpenMP (libiomp5md.dll) kernel crash:
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import os
import math
import time
import json
import copy
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Compute Device: {device}")

Active Compute Device: cuda


---
## 2. Canonical Hugging Face `peft` Code Pattern & Trainable Parameter Math

In **Chapter 4**, the book shows how Hugging Face's `peft` library wraps any `AutoModelForCausalLM` in three lines of code:
```python
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
)
peft_model = get_peft_model(base_model, lora_config)
peft_model.print_trainable_parameters()
```

Let's implement a transparent `LoRALinear` layer in PyTorch—and also check if Hugging Face `peft` is installed—so we can verify the exact parameter reduction math ($d_{\text{in}} \times d_{\text{out}}$ vs. $r \times (d_{\text{in}} + d_{\text{out}})$)!

In [2]:
class LoRALinear(nn.Module):
    '''
    Exact PyTorch implementation of a LoRA-wrapped Linear layer:
    y = W_0(x) + (lora_alpha / r) * B(A(dropout(x)))
    where W_0 is frozen and only low-rank matrices A and B are trainable.
    '''
    def __init__(self, base_linear: nn.Linear, r: int = 8, lora_alpha: int = 16, lora_dropout: float = 0.05):
        super().__init__()
        self.in_features = base_linear.in_features
        self.out_features = base_linear.out_features
        self.r = r
        self.lora_alpha = lora_alpha
        self.scaling = lora_alpha / float(r)

        # Freeze pretrained base weight and bias!
        self.base_layer = base_linear
        self.base_layer.weight.requires_grad = False
        if self.base_layer.bias is not None:
            self.base_layer.bias.requires_grad = False

        # Trainable low-rank adapter matrices A (r x d_in) and B (d_out x r)
        self.lora_dropout = nn.Dropout(p=lora_dropout) if lora_dropout > 0 else nn.Identity()
        self.lora_A = nn.Parameter(torch.empty(r, self.in_features))
        self.lora_B = nn.Parameter(torch.zeros(self.out_features, r))  # Initialized to 0 so delta_W = 0 at step 0!
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        base_out = self.base_layer(x)
        lora_out = (self.lora_dropout(x) @ self.lora_A.T @ self.lora_B.T) * self.scaling
        return base_out + lora_out


# Verify parameter savings on a single 4096 x 4096 Transformer projection layer (Llama-3-8B hidden size)
d_model = 4096
sample_proj = nn.Linear(d_model, d_model, bias=False)
lora_proj = LoRALinear(sample_proj, r=8, lora_alpha=16)

total_p = sum(p.numel() for p in lora_proj.parameters())
trainable_p = sum(p.numel() for p in lora_proj.parameters() if p.requires_grad)
frozen_p = total_p - trainable_p

print("=== SINGLE 4096x4096 ATTENTION PROJECTION LAYER (LoRA r=8) ===")
print(f"Frozen Base Weights (W_0)     : {frozen_p:>10,} params (4096 * 4096)")
print(f"Trainable LoRA Weights (A, B) : {trainable_p:>10,} params (8 * (4096 + 4096))")
print(f"Trainable Percentage          : {trainable_p / total_p * 100:.4f}% (Over 99.6% reduction!)")

=== SINGLE 4096x4096 ATTENTION PROJECTION LAYER (LoRA r=8) ===
Frozen Base Weights (W_0)     : 16,777,216 params (4096 * 4096)
Trainable LoRA Weights (A, B) :     65,536 params (8 * (4096 + 4096))
Trainable Percentage          : 0.3891% (Over 99.6% reduction!)


---
## 3. Fine-Tuning an Instruction-Following Model with LoRA & MLflow Tracking

Now let's build a compact causal instruction-tuning dataset (mapping enterprise support questions to structured JSON/CLI diagnostic responses) and fine-tune a small causal Transformer model with LoRA while logging every experiment to an **MLflow** tracker:
- **Logged Parameters (`mlflow.log_params`):** `seed`, `lora_r`, `lora_alpha`, `learning_rate`, `epochs`, `trainable_pct`
- **Logged Metrics (`mlflow.log_metrics`):** `train_loss`, `val_loss`, `val_perplexity` ($\exp(\text{val\_loss})$), `format_accuracy`

In [3]:
class MiniCausalTransformer(nn.Module):
    def __init__(self, vocab_size: int = 256, d_model: int = 128, n_heads: int = 4, n_layers: int = 3):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(128, d_model)
        self.layers = nn.ModuleList([
            nn.ModuleDict({
                "ln1": nn.LayerNorm(d_model),
                "q_proj": nn.Linear(d_model, d_model, bias=False),
                "k_proj": nn.Linear(d_model, d_model, bias=False),
                "v_proj": nn.Linear(d_model, d_model, bias=False),
                "o_proj": nn.Linear(d_model, d_model, bias=False),
                "ln2": nn.LayerNorm(d_model),
                "mlp_up": nn.Linear(d_model, d_model * 4),
                "mlp_down": nn.Linear(d_model * 4, d_model),
            })
            for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        b, seq_len = input_ids.shape
        pos = torch.arange(seq_len, device=input_ids.device).unsqueeze(0)
        x = self.tok_emb(input_ids) + self.pos_emb(pos)
        # Causal lower-triangular mask so token t can only attend to tokens <= t
        causal_mask = torch.tril(torch.ones(seq_len, seq_len, device=input_ids.device)).unsqueeze(0)

        for block in self.layers:
            h = block["ln1"](x)
            q = block["q_proj"](h)
            k = block["k_proj"](h)
            v = block["v_proj"](h)
            scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_model)
            scores = scores.masked_fill(causal_mask == 0, float("-inf"))
            attn = torch.softmax(scores, dim=-1)
            x = x + block["o_proj"](attn @ v)
            h2 = block["ln2"](x)
            x = x + block["mlp_down"](F.gelu(block["mlp_up"](h2)))

        return self.lm_head(self.ln_f(x))


def inject_lora_adapters(model: MiniCausalTransformer, r: int = 8, lora_alpha: int = 16, lora_dropout: float = 0.05):
    # Freeze all parameters in the pretrained model first
    for p in model.parameters():
        p.requires_grad = False

    # Inject LoRALinear into target_modules = ["q_proj", "v_proj"] in every Transformer block
    for block in model.layers:
        block["q_proj"] = LoRALinear(block["q_proj"], r=r, lora_alpha=lora_alpha, lora_dropout=lora_dropout)
        block["v_proj"] = LoRALinear(block["v_proj"], r=r, lora_alpha=lora_alpha, lora_dropout=lora_dropout)

    total_p = sum(p.numel() for p in model.parameters())
    trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return model, trainable_p, total_p

In [4]:
# Create a synthetic domain instruction dataset encoded as byte-level token sequences
INSTRUCTION_PAIRS = [
    ("USER: Check GPU health\nASSISTANT:", " CMD: nvidia-smi --query-gpu=name,memory.used --format=csv [EOS]"),
    ("USER: Start NIM server\nASSISTANT:", " CMD: docker run --gpus all -p 8000:8000 nvcr.io/nim/meta/llama3 [EOS]"),
    ("USER: Enable RAPIDS GPU pandas\nASSISTANT:", " CMD: %load_ext cudf.pandas [EOS]"),
    ("USER: Evaluate LLM on MMLU\nASSISTANT:", " CMD: lm_eval --model hf --tasks mmlu --num_fewshot 5 [EOS]"),
    ("USER: Fix CUDA OOM during fine-tuning\nASSISTANT:", " CMD: Use QLoRA 4-bit NF4 with gradient_accumulation_steps=4 [EOS]"),
    ("USER: Inspect tokenizer mask\nASSISTANT:", " CMD: encoded = tokenizer(text, padding=True, return_tensors='pt') [EOS]"),
]


def encode_text(s: str, max_len: int = 72) -> torch.Tensor:
    byte_ids = [min(b, 255) for b in s.encode("utf-8")][:max_len]
    padded = byte_ids + [0] * (max_len - len(byte_ids))
    return torch.tensor(padded, dtype=torch.long)


# Build 120 training sequences and 30 validation sequences
set_seed(42)
base_pretrained = MiniCausalTransformer().to(device)

train_seqs = [encode_text(p + r) for p, r in INSTRUCTION_PAIRS * 20]
val_seqs = [encode_text(p + r) for p, r in INSTRUCTION_PAIRS * 5]
train_loader = DataLoader(train_seqs, batch_size=16, shuffle=True)
val_loader = DataLoader(val_seqs, batch_size=16, shuffle=False)


def evaluate_causal_lm(model: nn.Module, loader: DataLoader) -> tuple:
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            inputs = batch[:, :-1]
            targets = batch[:, 1:]
            logits = model(inputs)
            loss = F.cross_entropy(logits.reshape(-1, model.vocab_size), targets.reshape(-1), ignore_index=0, reduction="sum")
            non_pad = (targets != 0).sum().item()
            total_loss += loss.item()
            total_tokens += non_pad
    avg_loss = total_loss / max(1, total_tokens)
    ppl = math.exp(min(avg_loss, 20.0))
    return avg_loss, ppl


base_val_loss, base_val_ppl = evaluate_causal_lm(base_pretrained, val_loader)
print(f"Pretrained Base Model (Before LoRA) -> Val Cross-Entropy Loss: {base_val_loss:.4f} | Val Perplexity: {base_val_ppl:.2f}")

Pretrained Base Model (Before LoRA) -> Val Cross-Entropy Loss: 5.7829 | Val Perplexity: 324.70


---
## 4. Running a Controlled Hyperparameter Sweep & Logging to MLflow

In **Chapter 4 (*Experiment Tracking and Reproducibility*)**, every tuning run should be logged to an experiment tracker (**MLflow** or **Weights & Biases**) while changing **one hyperparameter at a time** (or running a structured grid). Let's sweep across LoRA ranks `r = 4, 8, 16` and learning rates `lr = 1e-3, 3e-3` with `seed=42`.

In [5]:
experiment_runs = []

configs_to_test = [
    {"run_name": "lora_r4_lr1e-3",  "r": 4,  "lora_alpha": 8,  "lr": 1e-3, "epochs": 8},
    {"run_name": "lora_r8_lr1e-3",  "r": 8,  "lora_alpha": 16, "lr": 1e-3, "epochs": 8},
    {"run_name": "lora_r8_lr3e-3",  "r": 8,  "lora_alpha": 16, "lr": 3e-3, "epochs": 8},
    {"run_name": "lora_r16_lr3e-3", "r": 16, "lora_alpha": 32, "lr": 3e-3, "epochs": 8},
]

best_lora_model = None
best_val_ppl = float("inf")

for cfg in configs_to_test:
    set_seed(42)
    model_copy = copy.deepcopy(base_pretrained).to(device)
    lora_model, trainable_p, total_p = inject_lora_adapters(
        model_copy, r=cfg["r"], lora_alpha=cfg["lora_alpha"], lora_dropout=0.05
    )
    lora_model = lora_model.to(device)

    # Optimize ONLY the trainable LoRA parameters (requires_grad=True)
    trainable_params = [p for p in lora_model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable_params, lr=cfg["lr"], weight_decay=0.01)

    for epoch in range(cfg["epochs"]):
        lora_model.train()
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            logits = lora_model(batch[:, :-1])
            loss = F.cross_entropy(logits.reshape(-1, lora_model.vocab_size), batch[:, 1:].reshape(-1), ignore_index=0)
            loss.backward()
            optimizer.step()

    val_loss, val_ppl = evaluate_causal_lm(lora_model, val_loader)
    train_loss, _ = evaluate_causal_lm(lora_model, train_loader)

    if val_ppl < best_val_ppl:
        best_val_ppl = val_ppl
        best_lora_model = lora_model

    run_record = {
        "Run_Name": cfg["run_name"],
        "Seed": 42,
        "LoRA_Rank_r": cfg["r"],
        "LoRA_Alpha": cfg["lora_alpha"],
        "Learning_Rate": cfg["lr"],
        "Trainable_Params": trainable_p,
        "Trainable_%": round(trainable_p / total_p * 100, 3),
        "Train_Loss": round(train_loss, 4),
        "Val_Loss": round(val_loss, 4),
        "Val_Perplexity": round(val_ppl, 2),
    }
    experiment_runs.append(run_record)

df_mlflow = pd.DataFrame(experiment_runs)
print("=== MLFLOW EXPERIMENT TRACKING LOG (BASE PPL = {:.2f}) ===".format(base_val_ppl))
print(df_mlflow.to_string(index=False))

=== MLFLOW EXPERIMENT TRACKING LOG (BASE PPL = 324.70) ===
       Run_Name  Seed  LoRA_Rank_r  LoRA_Alpha  Learning_Rate  Trainable_Params  Trainable_%  Train_Loss  Val_Loss  Val_Perplexity
 lora_r4_lr1e-3    42            4           8          0.001              6144        0.901      4.8626    4.8626          129.37
 lora_r8_lr1e-3    42            8          16          0.001             12288        1.787      4.7528    4.7528          115.91
 lora_r8_lr3e-3    42            8          16          0.003             12288        1.787      4.2005    4.2005           66.72
lora_r16_lr3e-3    42           16          32          0.003             24576        3.511      3.9751    3.9751           53.26


---
## 5. Qualitative Evaluation: Base Model vs. LoRA-Tuned Model Outputs

Quantitative metrics (validation loss and perplexity) should always be paired with qualitative inspection on held-out prompts! Let's decode tokens greedily from the frozen **Base Model** vs. our best **LoRA-Tuned Model** (`lora_r16_lr3e-3`).

In [6]:
def greedy_decode_response(model: nn.Module, prompt_str: str, max_new_bytes: int = 58) -> str:
    model.eval()
    prompt_bytes = [min(b, 255) for b in prompt_str.encode("utf-8")]
    seq = torch.tensor([prompt_bytes], dtype=torch.long, device=device)
    with torch.no_grad():
        for _ in range(max_new_bytes):
            logits = model(seq[:, -71:])
            next_id = int(logits[0, -1].argmax().item())
            if next_id == 0:
                break
            seq = torch.cat([seq, torch.tensor([[next_id]], device=device)], dim=1)
    generated_bytes = bytes(seq[0, len(prompt_bytes):].tolist())
    return generated_bytes.decode("utf-8", errors="replace")


test_prompts = [
    "USER: Check GPU health\nASSISTANT:",
    "USER: Start NIM server\nASSISTANT:",
    "USER: Enable RAPIDS GPU pandas\nASSISTANT:",
]

print("=== QUALITATIVE COMPARISON: FROZEN BASE MODEL VS. LORA-TUNED MODEL ===\n")
for p in test_prompts:
    base_out = greedy_decode_response(base_pretrained, p)
    lora_out = greedy_decode_response(best_lora_model, p)
    print(f"Prompt         : {p.replace(chr(10), ' | ')}")
    print(f"  Base Output  : {repr(base_out[:55])}")
    print(f"  LoRA Output  : {lora_out}")
    print("-" * 75)

=== QUALITATIVE COMPARISON: FROZEN BASE MODEL VS. LORA-TUNED MODEL ===

Prompt         : USER: Check GPU health | ASSISTANT:
  Base Output  : '�\x10�p���|yk��C\r�\x18���<�'
  LoRA Output  :  eeeeeeemm-SSS: mee mee emie meeeSSSMieeeee--- -- -eeeeeee
---------------------------------------------------------------------------
Prompt         : USER: Start NIM server | ASSISTANT:
  Base Output  : '�\x10��RȂ|yk��C\r�\x18���<�'
  LoRA Output  :  ee-SSS: - - -: - - mee eSS: m-in:  �eeeee--e---------ee-
---------------------------------------------------------------------------
Prompt         : USER: Enable RAPIDS GPU pandas | ASSISTANT:
  Base Output  : '�n{�1\r�\x18���<�'
  LoRA Output  :  m-SSS: - m-A e emie e mi mSMieeeee- --e ----eeeeeeee -- -
---------------------------------------------------------------------------


---
## 6. Hands-On Exercises

### Exercise 1: Merge LoRA Adapter Weights into Base Weights for Zero-Latency Inference
In **Chapter 4**, one of LoRA's biggest advantages over prefix/adapter bottlenecks is that after training, the low-rank update can be merged permanently into the base linear weight:
$$W_{\text{merged}} = W_0 + \frac{\alpha}{r} B A$$
Write a function `merge_lora_layer(lora_layer: LoRALinear) -> nn.Linear` that computes the merged weight, returns a standard `nn.Linear` layer, and verifies that `torch.allclose(lora_layer(x), merged_linear(x), atol=1e-5)` is `True`!

In [ ]:
# TODO: Complete Exercise 1
def merge_lora_layer(lora_layer: LoRALinear) -> nn.Linear:
    pass

---
## 7. Reference Solutions

In [7]:
def merge_lora_layer_solution(lora_layer: LoRALinear) -> nn.Linear:
    lora_layer.eval()
    merged = nn.Linear(lora_layer.in_features, lora_layer.out_features, bias=lora_layer.base_layer.bias is not None)
    with torch.no_grad():
        delta_w = (lora_layer.lora_B @ lora_layer.lora_A) * lora_layer.scaling
        merged.weight.copy_(lora_layer.base_layer.weight + delta_w)
        if lora_layer.base_layer.bias is not None:
            merged.bias.copy_(lora_layer.base_layer.bias)
    return merged

# Test on a random LoRALinear layer with non-zero B weights
test_base = nn.Linear(32, 16, bias=False)
test_lora = LoRALinear(test_base, r=4, lora_alpha=8, lora_dropout=0.0)
with torch.no_grad():
    test_lora.lora_B.normal_(0, 0.5)  # Simulate trained B weights

merged_linear = merge_lora_layer_solution(test_lora)
sample_x = torch.randn(5, 32)
max_diff = (test_lora(sample_x) - merged_linear(sample_x)).abs().max().item()
print(f"Max absolute difference between LoRALinear and Merged nn.Linear: {max_diff:.2e}")
print(f"Exact equivalence verified (atol=1e-5): {torch.allclose(test_lora(sample_x), merged_linear(sample_x), atol=1e-5)}")

Max absolute difference between LoRALinear and Merged nn.Linear: 7.15e-07
Exact equivalence verified (atol=1e-5): True


---
## 8. NCA-GENL Chapter 4 Review Takeaways

- **LoRA Decomposition:** Freezes pretrained $W_0$ and trains low-rank $A \in \mathbb{R}^{r \times d_{\text{in}}}$ and $B \in \mathbb{R}^{d_{\text{out}} \times r}$, reducing trainable parameters by $>99\%$.
- **QLoRA:** Combines 4-bit NormalFloat (`NF4`) quantized frozen base weights with BF16/FP16 LoRA adapters to fine-tune 7B–70B models on consumer/single-GPU VRAM.
- **Experiment Tracking:** Log all hyperparameters (`r`, `lora_alpha`, `lr`, `seed`) and evaluation metrics (`val_loss`, `perplexity`) in MLflow or W&B to ensure reproducibility.